# Fine-tune MOESCORE Expert 1 trên Google Colab

Chọn **Runtime → Change runtime type → GPU** và **Runtime Version 2026.07 (Python 3.12)** trước khi chạy. Nếu có thể, chọn L4 hoặc A100; batch size 2 có thể vượt bộ nhớ của một số GPU T4.

Đưa các tệp sau lên Google Drive (sửa `PROJECT_DIR`/`DATA_ROOT` ở ô bên dưới nếu đặt nơi khác):

```text
MyDrive/MOESCORE/
├── Train_Expert1.py
├── config.yaml
├── loss/loss_function.py
├── weights/laion-clap.pt
└── data/
    ├── meta_data/train_average.csv
    ├── meta_data/validation_average.csv
    ├── wav/train/
    └── wav/validation/
```

CSV cần có các cột `wav_file_name`, `text`, `average_score`. `wav_file_name` phải trỏ tới tệp WAV tương ứng trong thư mục train/validation. Checkpoint tốt nhất được ghi vào Drive để còn sau khi Colab ngắt phiên.

In [ ]:
import sys
assert sys.version_info[:2] == (3, 12), (
    f'Runtime đang dùng Python {sys.version.split()[0]}. '
    'Chọn Runtime Version 2026.07 (Python 3.12) trong Change runtime type rồi chạy lại.'
)
print('Python:', sys.version.split()[0])

In [ ]:
%pip install --no-cache-dir 'numpy==1.26.4' 'laion-clap==1.1.7' 'transformers>=4.51.3,<5' 'pyyaml'

Nếu `pip` báo `Successfully installed` nhưng liệt kê xung đột với các gói Colab cài sẵn (JAX, CuPy, OpenCV, Gradio, ...), hãy tiếp tục theo bước sau. Expert 1 không dùng trực tiếp các gói đó; LAION-CLAP 1.1.7 cần NumPy < 2.

**Sau khi cài đặt:** chọn **Runtime → Restart session/runtime**, rồi chạy tiếp từ ô kiểm tra import bên dưới. Không chạy lại ô cài đặt sau khi khởi động lại. Bước này loại bỏ các thư viện NumPy cũ đã nạp trong kernel. Nếu ô kiểm tra import vẫn lỗi, cần xem traceback mới trước khi chạy huấn luyện.

In [ ]:
import sys
import numpy as np
import pandas as pd
import scipy
import torch
import torchaudio
import laion_clap

print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__)
assert np.__version__ == '1.26.4', 'NumPy chưa đúng phiên bản; kiểm tra ô cài đặt và restart runtime.'
print('pandas:', pd.__version__)
print('SciPy:', scipy.__version__)
print('PyTorch:', torch.__version__)
print('torchaudio:', torchaudio.__version__)
print('CUDA:', torch.version.cuda)
assert torch.cuda.is_available(), 'Chưa có GPU: chọn GPU trong Runtime → Change runtime type.'
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM (GiB):', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))

In [ ]:
from google.colab import drive
from pathlib import Path
import pandas as pd

drive.mount('/content/drive')
PROJECT_DIR = Path('/content/drive/MyDrive/MOESCORE')
DATA_ROOT = PROJECT_DIR / 'data'
TRAIN_CSV = DATA_ROOT / 'meta_data/train_average.csv'
VAL_CSV = DATA_ROOT / 'meta_data/validation_average.csv'
CHECKPOINT = PROJECT_DIR / 'weights/laion-clap.pt'
OUTPUT = PROJECT_DIR / 'weights/expert1_best.pt'

required = [
    PROJECT_DIR / 'Train_Expert1.py',
    PROJECT_DIR / 'config.yaml',
    PROJECT_DIR / 'loss/loss_function.py',
    CHECKPOINT,
    TRAIN_CSV,
    VAL_CSV,
    DATA_ROOT / 'wav/train',
    DATA_ROOT / 'wav/validation',
]
missing = [str(path) for path in required if not path.exists()]
assert not missing, 'Thiếu tệp/thư mục:\n' + '\n'.join(missing)

for split, csv_path in [('train', TRAIN_CSV), ('validation', VAL_CSV)]:
    frame = pd.read_csv(csv_path)
    columns = {'wav_file_name', 'text', 'average_score'}
    assert columns <= set(frame.columns), f'{csv_path.name} thiếu cột {columns - set(frame.columns)}'
    assert len(frame) >= 2, f'{csv_path.name} cần ít nhất 2 mẫu'
    if split == 'validation':
        assert frame['average_score'].nunique() > 1, 'validation.csv cần ít nhất 2 giá trị score khác nhau để tính SRCC'
    wav_dir = DATA_ROOT / 'wav' / split
    first_wav = wav_dir / str(frame.iloc[0]['wav_file_name']).lstrip('/')
    assert first_wav.is_file(), f'Không tìm thấy WAV đầu tiên: {first_wav}'
    print(f'{split}: {len(frame)} mẫu; WAV đầu tiên: {first_wav.name}')
print('LAION-CLAP:', round(CHECKPOINT.stat().st_size / 2**30, 2), 'GiB')

In [ ]:
import subprocess

BATCH_SIZE = 2
EPOCHS = 50
PATIENCE = 10
WARMUP_RATIO = 0.1

command = [
    sys.executable, '-u', 'Train_Expert1.py',
    '--data-root', str(DATA_ROOT),
    '--train-csv', str(TRAIN_CSV),
    '--val-csv', str(VAL_CSV),
    '--checkpoint', str(CHECKPOINT),
    '--config', str(PROJECT_DIR / 'config.yaml'),
    '--output', str(OUTPUT),
    '--device', 'cuda',
    '--batch-size', str(BATCH_SIZE),
    '--epochs', str(EPOCHS),
    '--warmup-ratio', str(WARMUP_RATIO),
    '--patience', str(PATIENCE),
    '--num-workers', '0',
    '--log-every', '100',
]
subprocess.run(command, cwd=PROJECT_DIR, check=True)

In [ ]:
assert OUTPUT.is_file(), f'Không thấy checkpoint: {OUTPUT}'
print('Checkpoint Expert 1:', OUTPUT)
print('Kích thước (GiB):', round(OUTPUT.stat().st_size / 2**30, 2))